Load all 10 CSV files data Info

In [4]:
import pandas as pd
import requests
from pathlib import Path
import os


In [5]:
Data_path="../data/raw"

csv_files=[f for f in os.listdir(Data_path) if f.endswith(".csv")]
for file in csv_files:
    print("\n"+"="*50)

    print(f"Dataset: {file}")

    df=pd.read_csv(os.path.join(Data_path,file))

    print("Shape: ",df.shape)

    print("Types: ",df.dtypes)

    print("Missing values: ",df.isnull().sum())





Dataset: 01_fund_master.csv
Shape:  (40, 15)
Types:  amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object
Missing values:  amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

Dataset: 02_nav_history.csv
Shape:  (46000, 3)
Types:  amfi_code      int64
date          o

Fetching live NAV data from mfapi

In [8]:
Funds={
    "HDFC Top 100":125497,
    "SBI Bluechip":119551,
    "ICICI Bluechip":120503,
    "Nippon Large Cap":118632,
    "Axis Bluechip":119092,
    "Kotak Bluechip":120841
}
try:
    for funds_name,scheme_code in Funds.items():
        url=f'https://api.mfapi.in/mf/{scheme_code}'
        response=requests.get(url,timeout=30)

        response.raise_for_status()

        json_data=response.json()

        nav_df=pd.DataFrame(json_data["data"])

        output_file=os.path.join(Data_path,f"{funds_name}_NAV.csv")

        nav_df.to_csv(output_file,index=False)

        print("="*60)

        print(f'Fund: {funds_name}')
        print(f'Scheme Code: {scheme_code}')
        print(f'Record Saved :{len(nav_df)}')
        print(f'File: {output_file}')
except Exception as e:
    print(e)

Fund: HDFC Top 100
Scheme Code: 125497
Record Saved :3091
File: ../data/raw\HDFC Top 100_NAV.csv
Fund: SBI Bluechip
Scheme Code: 119551
Record Saved :3236
File: ../data/raw\SBI Bluechip_NAV.csv
Fund: ICICI Bluechip
Scheme Code: 120503
Record Saved :3307
File: ../data/raw\ICICI Bluechip_NAV.csv
Fund: Nippon Large Cap
Scheme Code: 118632
Record Saved :3298
File: ../data/raw\Nippon Large Cap_NAV.csv
Fund: Axis Bluechip
Scheme Code: 119092
Record Saved :3565
File: ../data/raw\Axis Bluechip_NAV.csv
Fund: Kotak Bluechip
Scheme Code: 120841
Record Saved :3301
File: ../data/raw\Kotak Bluechip_NAV.csv


In [15]:
df=pd.read_csv(os.path.join(Data_path,"01_fund_master.csv"))
print("Unique Fund Houses: ",df["fund_house"].nunique)
print("Unique Fund Houses: ",df["fund_house"].unique)

Unique Fund Houses:  <bound method IndexOpsMixin.nunique of 0              SBI Mutual Fund
1              SBI Mutual Fund
2              SBI Mutual Fund
3              SBI Mutual Fund
4              SBI Mutual Fund
5             HDFC Mutual Fund
6             HDFC Mutual Fund
7             HDFC Mutual Fund
8             HDFC Mutual Fund
9             HDFC Mutual Fund
10         ICICI Prudential MF
11         ICICI Prudential MF
12         ICICI Prudential MF
13         ICICI Prudential MF
14         ICICI Prudential MF
15             Nippon India MF
16             Nippon India MF
17             Nippon India MF
18             Nippon India MF
19             Nippon India MF
20           Kotak Mahindra MF
21           Kotak Mahindra MF
22           Kotak Mahindra MF
23           Kotak Mahindra MF
24            Axis Mutual Fund
25            Axis Mutual Fund
26            Axis Mutual Fund
27            Axis Mutual Fund
28    Aditya Birla Sun Life MF
29    Aditya Birla Sun Life MF
30    Adit

In [17]:
print("Categories: ")
print(df["category"].value_counts())

Categories: 
category
Equity    34
Debt       6
Name: count, dtype: int64


In [18]:
print("Risk Categories: \n",df["risk_category"].value_counts())

Risk Categories: 
 risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64


In [20]:
fund_master_code=set(df["amfi_code"])
ndf=pd.read_csv(os.path.join(Data_path,"02_nav_history.csv"))
nav_master_code=set(ndf["amfi_code"])
missing_code=fund_master_code-nav_master_code
print("="*50)
print("Fund Master Codes :", len(fund_master_code))
print("NAV History Codes :", len(nav_master_code))
print("Missing Codes     :", len(missing_code))
print("Missing Code List :", missing_code)

Fund Master Codes : 40
NAV History Codes : 40
Missing Codes     : 0
Missing Code List : set()
